In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day03"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


**샘플 파일 조회**

In [2]:
if not SAMPLES.is_dir():
    print('재료 폴더가 없습니다')
else:
    all_files = [p for p in SAMPLES.rglob('*') if p.is_file()]
    top_files = [p for p in SAMPLES.iterdir() if p.is_file()]
    sub_files = [p for p in (SAMPLES / '_formats').rglob('*') if p.is_file()]
    today = sorted((p.name for p in SAMPLES.glob('DOC-HR-014_*_v2.0.*')))
    print(f'재료 파일 : {len(all_files)}개')
    print(f'  samples 바로 아래 : {len(top_files)}개 · _formats 안 : {len(sub_files)}개')
    print('오늘 읽을 세 형식 :')
    for name in today:
        print(' ', name)

재료 파일 : 30개
  samples 바로 아래 : 24개 · _formats 안 : 6개
오늘 읽을 세 형식 :
  DOC-HR-014_국내출장_여비_규정_v2.0.docx
  DOC-HR-014_국내출장_여비_규정_v2.0.hwpx
  DOC-HR-014_국내출장_여비_규정_v2.0.pdf


**로컬 파서가 리턴해주는 데이터 확인**

In [3]:
from app.rag.local_parsers import parse_local
TARGET = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(TARGET)
print(f'블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')

블록 233 · 표 7 · 쪽 1


**파싱한 블럭 하나가 어떤식으로 데이터를 갖고 있는지 확인**

In [4]:
for i, b in enumerate(doc.blocks[:5]):
    head = b.text[:36] + ('…' if len(b.text) > 36 else '')
    print(f'[{i:2d}] {b.kind} | {b.locator} | {head}')
first_table = next((b for b in doc.blocks if b.kind == '표'))
print()
print('첫 표 블록 : locator=' + first_table.locator)
print(' ', first_table.text[:60])

[ 0] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 일반
[ 1] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 국내출장 여비 규정
[ 2] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | v2.0
[ 3] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 문서번호  DOC-HR-014
[ 4] 조항 | DOC-HR-014_국내출장_여비_규정_v2.0 | 시행일    2025-07-01

첫 표 블록 : locator=표1
  | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|--


**반례 : 문서를 통째로 하나 넣으면?**

In [5]:
joined = '\n'.join((b.text for b in doc.blocks))
print(joined[4000:4200])

 기준으로 한다.
② 3급 이상의 직원은 업무 수행상 필요한 경우 특실을 이용할 수 있다.
③ 심야 이동으로 침대차를 이용한 경우에는 그 요금을 지급하고 숙박비를 지급하지 아니한다.
제19조(항공기)
① 국내선 항공기는 철도 대비 이동 시간이 2시간 이상 단축되는 경우에 한하여 이용할 수 있다.
② 국내선 항공기의 좌석 등급은 일반석으로 한다.
③ 항공기를


**반례 : 블럭 한개만 전달**

In [ ]:
# 숙박비가 포함된 블럭 조회
hits = [(i, b) for i, b in enumerate(doc.blocks) if '숙박비' in b.text]
print(f"'숙박비' 가 들어 있는 블록 : {len(hits)}개")
for i, b in hits[:5]:
    line = b.text.splitlines()[0]
    print(f'  [{i:3d}] {len(b.text):3d}자 | {line[:44]}')
print(f'  ... (나머지 {len(hits) - 5}개는 생략)')

# '광역시 숙박비가 얼마까지 되나요?'
picked = next((b for _, b in hits if b.text.startswith(('①', '②')) and '광역시' in b.text))
print()
print('고른 블록 :', picked.text)
print('  이 블록 안에 조 표시가 있나 :', '제' in picked.text)
short = [b for b in doc.blocks if len(b.text) < 30]
print(f'  30자도 안 되는 블록 : {len(short)}개 / {len(doc.blocks)}개')


'숙박비' 가 들어 있는 블록 : 27개
  [  9] 291자 | | 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
  [ 29]   7자 | 제2절 숙박비
  [ 30]   9자 | 제14조(숙박비)
  [ 32]  13자 | 제16조(숙박비의 예외)
  [ 68]  39자 | 3. "여비"란 출장에 소요되는 일비·식비·숙박비 및 교통비를 말한다.
  ... (나머지 22개는 생략)

고른 블록 : ② 광역시 지역의 숙박비는 1박당 70,000원을 상한으로 한다.
  이 블록 안에 조 표시가 있나 : False
  30자도 안 되는 블록 : 131개 / 233개


**두 문장의 글자 겹침을 0 ~ 1 사이로 재보기**

In [7]:
def jaccard(x: str, y: str) -> float:
    sx, sy = (set(x.replace(' ', '')), set(y.replace(' ', '')))
    return len(sx & sy) / len(sx | sy)
PAIRS = [('광역시 숙박비 상한', '잠자리 비용 한도'), ('광역시 숙박비 상한', '광역시 숙박비 상한액')]
for left, right in PAIRS:
    shared = sorted(set(left.replace(' ', '')) & set(right.replace(' ', '')))
    shown = shared if len(shared) <= 3 else f'{len(shared)}개'
    print(f"'{left}' vs '{right}'  겹치는 글자 {shown} · 자카드 {jaccard(left, right):.3f}")

'광역시 숙박비 상한' vs '잠자리 비용 한도'  겹치는 글자 ['비', '한'] · 자카드 0.154
'광역시 숙박비 상한' vs '광역시 숙박비 상한액'  겹치는 글자 8개 · 자카드 0.889


**낱말이 겹쳐야 찾게되는 검색을 흉내내보기**

In [8]:
from app.rag.local_parsers import parse_local

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
doc = parse_local(DOCX)

print('연 문서 :', DOCX.name, '· 블록', len(doc.blocks), '개')
print()

def like(word: str) -> int:
    return sum((1 for b in doc.blocks if word in b.text))

for word in ['숙박비', '잠자리', '잠자리 비용']:
    print(f"LIKE '%{word}%' → {like(word)}건")


연 문서 : DOC-HR-014_국내출장_여비_규정_v2.0.docx · 블록 233 개

LIKE '%숙박비%' → 27건
LIKE '%잠자리%' → 0건
LIKE '%잠자리 비용%' → 0건


**예시로 만든 2차원 벡터로 모델 사용 방식 테스트**

In [9]:
import math
POINTS = {'광역시 숙박비 상한': [0.9, 0.2], '잠자리 비용 한도': [0.86, 0.28], '출장 신청 절차': [0.1, 0.95]}

def distance(p: list[float], q: list[float]) -> float:
    return math.sqrt(sum(((a - b) ** 2 for a, b in zip(p, q))))
    
BASE = '광역시 숙박비 상한'
print('기준 :', BASE, POINTS[BASE])

for text, point in POINTS.items():
    if text == BASE:
        continue
    gap = distance(POINTS[BASE], point)
    print(f"  {text}  {point} → 거리 {gap:.3f}   {('← 가깝다' if gap < 0.5 else '← 멀다')}")


기준 : 광역시 숙박비 상한 [0.9, 0.2]
  잠자리 비용 한도  [0.86, 0.28] → 거리 0.089   ← 가깝다
  출장 신청 절차  [0.1, 0.95] → 거리 1.097   ← 멀다


**차원 dimension  : 벡터를 이루는 실수의 개수**

In [10]:
v2 = [0.9, 0.2]
v3 = [0.6, 0.8, 0.0]
v1024 = [round(math.sin(i), 4) for i in range(1024)]


def size(v: list[float]) -> float:
    return math.sqrt(sum(x * x for x in v))


for label, v in [("2차원", v2), ("3차원", v3), ("1024차원", v1024)]:
    # 문법 - v[:5] 는 앞에서 다섯 개만 잘라 낸 새 리스트다(슬라이싱)
    print(f"{label:<8} : 길이 {len(v):<4} 앞 5개 {v[:5]}")

print()
print(f"크기(‖v‖) : {size(v2):.4f} · {size(v3):.4f} · {size(v1024):.4f}")

2차원      : 길이 2    앞 5개 [0.9, 0.2]
3차원      : 길이 3    앞 5개 [0.6, 0.8, 0.0]
1024차원   : 길이 1024 앞 5개 [0.0, 0.8415, 0.9093, 0.1411, -0.7568]

크기(‖v‖) : 0.9220 · 1.0000 · 22.6283
